# GSS Full Dataset Exploratory Data Analysis

This notebook explores the complete General Social Survey (GSS) dataset from 1972-2024, focusing on variables related to job satisfaction, mental health, work attitudes, and discrimination.

## Variables of Interest

### Outcome
- **SATJOB**: Job satisfaction (1=Very satisfied, 2=Moderately satisfied, 3=A little dissatisfied, 4=Very dissatisfied)

### Mental Health (PHQ-4 items)
- **HLTHDEP**: Health depression scale (1-5)
- **FEELDOWN**: Over the last 2 weeks, how often have you been bothered by feeling down, depressed, or hopeless (1-4)
- **NOINTEREST**: Over the last 2 weeks, how often have you been bothered by little interest or pleasure in doing things (1-4)
- **FEELNERV**: Over the last 2 weeks, how often have you been bothered by feeling nervous, anxious, or on edge (1-4)
- **WORRY**: Over the last 2 weeks, how often have you been bothered by not being able to stop or control worrying (1-4)

### Work Attitudes
- **WRKMEANGFL**: "I find my work to be very meaningful" (1=Strongly agree to 4=Strongly disagree)
- **STRESS**: How often do you find your work stressful?

### Discrimination Perceptions
- **DISCAFFWNV**: Chances a woman won't get a job/promotion while an equally/less qualified man gets one (1=Very likely to 4=Very unlikely)
- **DISCAFFMV**: Chances a man won't get a job/promotion while an equally/less qualified woman gets one
- **WKAGEISM**: Do you feel discriminated against on your job because of your age?
- **WKRACISM**: Do you feel discriminated against on your job because of your race or ethnic origin?
- **WKSEXISM**: Do you feel discriminated against on your job because of your gender?

### Socioeconomic
- **FINRELA**: Family income compared to American families in general (1=Far below average to 5=Far above average)
- **EDUC**: Years of education

### Demographics
- **AGE**: Age in years
- **SEX**: 1=Male, 2=Female

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
from pathlib import Path

# Configure plotly
pio.renderers.default = "notebook_connected"

# Display settings
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)

print("Libraries imported successfully")

Libraries imported successfully


## 1. Data Loading

In [ ]:
# Load the parquet file
data_path = Path('../data/GSS_stata/gss7224_r2.parquet')
print(f"Loading data from: {data_path}")

gss = pd.read_parquet(data_path)

print(f"\nDataset loaded successfully!")
print(f"Shape: {gss.shape[0]:,} rows x {gss.shape[1]:,} columns")
print(f"Memory usage: {gss.memory_usage(deep=True).sum() / 1e6:.1f} MB")

# Year range
if 'year' in gss.columns:
    print(f"Years: {int(gss['year'].min())} - {int(gss['year'].max())}")

In [3]:
# Define our variables of interest
target_vars = {
    # Outcome
    'satjob': 'Job Satisfaction (1-4)',
    
    # Mental Health
    'hlthdep': 'Depression Scale (1-5)',
    'feeldown': 'Feeling Down (1-4)',
    'nointerest': 'No Interest (1-4)',
    'feelnerv': 'Feeling Nervous (1-4)',
    'worry': 'Worry (1-4)',
    
    # Work Attitudes
    'wrkmeangfl': 'Work Meaningful (1-4)',
    'stress': 'Work Stress',
    
    # Discrimination
    'discaffwnv': 'Discrim Against Women (1-4)',
    'discaffmv': 'Discrim Against Men (1-4)',
    'wkageism': 'Age Discrimination',
    'wkracism': 'Race Discrimination',
    'wksexism': 'Sex Discrimination',
    
    # Socioeconomic
    'finrela': 'Relative Income (1-5)',
    'educ': 'Years of Education',
    
    # Demographics
    'age': 'Age',
    'sex': 'Sex (1=M, 2=F)',
    'year': 'Survey Year'
}

# Check which variables exist in the dataset
available_vars = {}
missing_vars = []

for var, desc in target_vars.items():
    if var in gss.columns:
        available_vars[var] = desc
    else:
        missing_vars.append(var)

print(f"Variables found: {len(available_vars)}/{len(target_vars)}")
print(f"\nAvailable: {list(available_vars.keys())}")
if missing_vars:
    print(f"\nNot found in dataset: {missing_vars}")

Variables found: 17/18

Available: ['satjob', 'hlthdep', 'feeldown', 'nointerest', 'feelnerv', 'worry', 'wrkmeangfl', 'stress', 'discaffwnv', 'wkageism', 'wkracism', 'wksexism', 'finrela', 'educ', 'age', 'sex', 'year']

Not found in dataset: ['discaffmv']


## 2. Missing Data Analysis

In [4]:
# Missing data for target variables
vars_to_check = list(available_vars.keys())

missing_data = []
for var in vars_to_check:
    n_total = len(gss)
    n_missing = gss[var].isna().sum()
    n_available = n_total - n_missing
    pct_available = n_available / n_total * 100
    
    missing_data.append({
        'Variable': var,
        'Description': available_vars[var],
        'Available': n_available,
        'Missing': n_missing,
        'Pct Available': pct_available
    })

missing_df = pd.DataFrame(missing_data).sort_values('Pct Available', ascending=False)

print("Missing Data Summary for Target Variables:")
print("="*80)
for _, row in missing_df.iterrows():
    print(f"{row['Variable']:<15} {row['Description']:<30} {row['Available']:>7,} ({row['Pct Available']:>5.1f}%)")

Missing Data Summary for Target Variables:
year            Survey Year                     75,699 (100.0%)
sex             Sex (1=M, 2=F)                  75,568 ( 99.8%)
educ            Years of Education              75,413 ( 99.6%)
age             Age                             74,829 ( 98.9%)
finrela         Relative Income (1-5)           70,580 ( 93.2%)
satjob          Job Satisfaction (1-4)          54,131 ( 71.5%)
stress          Work Stress                     15,879 ( 21.0%)
wkageism        Age Discrimination              10,408 ( 13.7%)
wkracism        Race Discrimination             10,399 ( 13.7%)
wksexism        Sex Discrimination               9,251 ( 12.2%)
discaffwnv      Discrim Against Women (1-4)      2,554 (  3.4%)
wrkmeangfl      Work Meaningful (1-4)            1,953 (  2.6%)
feeldown        Feeling Down (1-4)               1,942 (  2.6%)
feelnerv        Feeling Nervous (1-4)            1,942 (  2.6%)
nointerest      No Interest (1-4)                1,942 (  2.6

In [5]:
# Visualize missing data
fig = px.bar(missing_df, x='Variable', y='Pct Available',
             title='Data Availability for Target Variables',
             labels={'Pct Available': 'Percent Available', 'Variable': 'Variable'},
             color='Pct Available',
             color_continuous_scale='RdYlGn')
fig.update_layout(xaxis_tickangle=-45, showlegend=False)
fig.add_hline(y=50, line_dash='dash', line_color='red', annotation_text='50%')
fig.show()

## 3. Variable Availability by Year

GSS asks different questions in different years. Understanding when each variable was measured is crucial.

In [6]:
# Check which years each variable has data
print("Years with Data for Each Variable:")
print("="*70)

var_years = {}
for var in vars_to_check:
    if var == 'year':
        continue
    years_with_data = gss.loc[gss[var].notna(), 'year'].dropna().unique()
    years_with_data = sorted(years_with_data)
    var_years[var] = years_with_data
    
    if len(years_with_data) > 0:
        if len(years_with_data) <= 5:
            year_str = ', '.join([str(int(y)) for y in years_with_data])
        else:
            year_str = f"{int(min(years_with_data))}-{int(max(years_with_data))} ({len(years_with_data)} years)"
        print(f"{var:<15} {available_vars[var]:<30} {year_str}")
    else:
        print(f"{var:<15} {available_vars[var]:<30} No data")

Years with Data for Each Variable:
satjob          Job Satisfaction (1-4)         1972-2024 (35 years)
hlthdep         Depression Scale (1-5)         2022
feeldown        Feeling Down (1-4)             2022
nointerest      No Interest (1-4)              2022
feelnerv        Feeling Nervous (1-4)          2022
worry           Worry (1-4)                    2022
wrkmeangfl      Work Meaningful (1-4)          2022
stress          Work Stress                    1989-2022 (11 years)
discaffwnv      Discrim Against Women (1-4)    2021, 2022, 2024
wkageism        Age Discrimination             2002-2022 (7 years)
wkracism        Race Discrimination            2002-2022 (7 years)
wksexism        Sex Discrimination             2002-2022 (6 years)
finrela         Relative Income (1-5)          1972-2024 (35 years)
educ            Years of Education             1972-2024 (35 years)
age             Age                            1972-2024 (35 years)
sex             Sex (1=M, 2=F)                 1

In [7]:
# Create availability heatmap by year
analysis_vars = [v for v in vars_to_check if v != 'year']
years = sorted(gss['year'].dropna().unique())

# Build availability matrix
avail_matrix = []
for var in analysis_vars:
    row = []
    for year in years:
        count = gss.loc[gss['year'] == year, var].notna().sum()
        row.append(count)
    avail_matrix.append(row)

avail_array = np.array(avail_matrix)

fig = px.imshow(avail_array,
                x=[int(y) for y in years],
                y=analysis_vars,
                labels=dict(x='Year', y='Variable', color='N Responses'),
                title='Variable Availability by Survey Year',
                aspect='auto',
                color_continuous_scale='Blues')
fig.update_layout(height=500)
fig.show()

In [8]:
# Calculate pairwise data availability (for regression analysis)
print("Pairwise Complete Cases with SATJOB:")
print("="*60)

if 'satjob' in available_vars:
    for var in analysis_vars:
        if var != 'satjob':
            complete = (gss['satjob'].notna() & gss[var].notna()).sum()
            print(f"SATJOB + {var:<15}: {complete:>6,} complete pairs")

Pairwise Complete Cases with SATJOB:
SATJOB + hlthdep        :    751 complete pairs
SATJOB + feeldown       :  1,936 complete pairs
SATJOB + nointerest     :  1,937 complete pairs
SATJOB + feelnerv       :  1,937 complete pairs
SATJOB + worry          :  1,934 complete pairs
SATJOB + wrkmeangfl     :  1,950 complete pairs
SATJOB + stress         : 13,289 complete pairs
SATJOB + discaffwnv     :  1,793 complete pairs
SATJOB + wkageism       :  9,520 complete pairs
SATJOB + wkracism       :  9,510 complete pairs
SATJOB + wksexism       :  8,363 complete pairs
SATJOB + finrela        : 53,338 complete pairs
SATJOB + educ           : 53,955 complete pairs
SATJOB + age            : 53,568 complete pairs
SATJOB + sex            : 54,035 complete pairs


## 4. Variable Distributions

In [9]:
# Job Satisfaction Distribution
if 'satjob' in available_vars:
    satjob_counts = gss['satjob'].value_counts().sort_index()
    labels = {1: 'Very Satisfied', 2: 'Mod. Satisfied', 3: 'A Little Dissatisfied', 4: 'Very Dissatisfied'}
    
    fig = px.bar(x=[labels.get(int(x), x) for x in satjob_counts.index],
                 y=satjob_counts.values,
                 title='Job Satisfaction Distribution (SATJOB)',
                 labels={'x': 'Job Satisfaction', 'y': 'Count'})
    fig.show()
    
    print("Job Satisfaction Statistics:")
    print(gss['satjob'].describe())

Job Satisfaction Statistics:
count    54131.000000
mean         1.708208
std          0.810541
min          1.000000
25%          1.000000
50%          2.000000
75%          2.000000
max          4.000000
Name: satjob, dtype: float64


In [10]:
# Mental Health Variables Distribution
mh_vars = ['hlthdep', 'feeldown', 'nointerest', 'feelnerv', 'worry']
mh_vars = [v for v in mh_vars if v in available_vars]

if mh_vars:
    n_vars = len(mh_vars)
    cols = min(3, n_vars)
    rows = (n_vars + cols - 1) // cols
    
    fig = make_subplots(rows=rows, cols=cols, 
                        subplot_titles=[available_vars[v] for v in mh_vars])
    
    for i, var in enumerate(mh_vars):
        r, c = i // cols + 1, i % cols + 1
        counts = gss[var].value_counts().sort_index()
        fig.add_trace(
            go.Bar(x=[str(int(x)) for x in counts.index], y=counts.values, 
                   name=var, showlegend=False),
            row=r, col=c
        )
    
    fig.update_layout(height=300*rows, title='Mental Health Variables Distribution')
    fig.show()

In [11]:
# Work Attitudes Distribution
work_vars = ['wrkmeangfl', 'stress']
work_vars = [v for v in work_vars if v in available_vars]

if work_vars:
    fig = make_subplots(rows=1, cols=len(work_vars),
                        subplot_titles=[available_vars[v] for v in work_vars])
    
    for i, var in enumerate(work_vars):
        counts = gss[var].value_counts().sort_index()
        fig.add_trace(
            go.Bar(x=[str(x) for x in counts.index], y=counts.values,
                   name=var, showlegend=False),
            row=1, col=i+1
        )
    
    fig.update_layout(height=400, title='Work Attitudes Distribution')
    fig.show()

In [12]:
# Discrimination Variables Distribution
discrim_vars = ['discaffwnv', 'discaffmv', 'wkageism', 'wkracism', 'wksexism']
discrim_vars = [v for v in discrim_vars if v in available_vars]

if discrim_vars:
    n_vars = len(discrim_vars)
    cols = min(3, n_vars)
    rows = (n_vars + cols - 1) // cols
    
    fig = make_subplots(rows=rows, cols=cols,
                        subplot_titles=[available_vars[v] for v in discrim_vars])
    
    for i, var in enumerate(discrim_vars):
        r, c = i // cols + 1, i % cols + 1
        counts = gss[var].value_counts().sort_index()
        fig.add_trace(
            go.Bar(x=[str(x) for x in counts.index], y=counts.values,
                   name=var, showlegend=False),
            row=r, col=c
        )
    
    fig.update_layout(height=300*rows, title='Discrimination Variables Distribution')
    fig.show()

In [13]:
# Socioeconomic Variables
se_vars = ['finrela', 'educ']
se_vars = [v for v in se_vars if v in available_vars]

if se_vars:
    fig = make_subplots(rows=1, cols=len(se_vars),
                        subplot_titles=[available_vars[v] for v in se_vars])
    
    for i, var in enumerate(se_vars):
        data = gss[var].dropna()
        if var == 'finrela':
            counts = data.value_counts().sort_index()
            labels = {1: 'Far Below', 2: 'Below', 3: 'Average', 4: 'Above', 5: 'Far Above'}
            fig.add_trace(
                go.Bar(x=[labels.get(int(x), str(x)) for x in counts.index], 
                       y=counts.values, showlegend=False),
                row=1, col=i+1
            )
        else:
            fig.add_trace(
                go.Histogram(x=data, nbinsx=21, showlegend=False),
                row=1, col=i+1
            )
    
    fig.update_layout(height=400, title='Socioeconomic Variables Distribution')
    fig.show()
    
    # Statistics
    for var in se_vars:
        print(f"\n{var.upper()} Statistics:")
        print(gss[var].describe())


FINRELA Statistics:
count    70580.000000
mean         2.872018
std          0.869721
min          1.000000
25%          2.000000
50%          3.000000
75%          3.000000
max          5.000000
Name: finrela, dtype: float64

EDUC Statistics:
count    75413.000000
mean        13.086219
std          3.180636
min          0.000000
25%         12.000000
50%         13.000000
75%         16.000000
max         20.000000
Name: educ, dtype: float64


In [14]:
# Demographics
fig = make_subplots(rows=1, cols=2, subplot_titles=['Age Distribution', 'Sex Distribution'])

if 'age' in available_vars:
    fig.add_trace(
        go.Histogram(x=gss['age'].dropna(), nbinsx=50, showlegend=False),
        row=1, col=1
    )

if 'sex' in available_vars:
    sex_counts = gss['sex'].value_counts().sort_index()
    fig.add_trace(
        go.Bar(x=['Male', 'Female'], y=sex_counts.values, showlegend=False),
        row=1, col=2
    )

fig.update_layout(height=400, title='Demographic Distributions')
fig.show()

## 5. Trends Over Time

In [15]:
# Job Satisfaction over time
if 'satjob' in available_vars and 'year' in gss.columns:
    satjob_by_year = gss.groupby('year')['satjob'].agg(['mean', 'std', 'count']).reset_index()
    satjob_by_year = satjob_by_year[satjob_by_year['count'] >= 100]  # Filter low-n years
    
    fig = px.line(satjob_by_year, x='year', y='mean',
                  title='Average Job Satisfaction Over Time (lower = more satisfied)',
                  labels={'year': 'Year', 'mean': 'Average Job Satisfaction'},
                  markers=True)
    fig.show()
    
    print(f"Years with data: {len(satjob_by_year)}")
    print(f"Range: {satjob_by_year['mean'].min():.3f} - {satjob_by_year['mean'].max():.3f}")

Years with data: 35
Range: 1.631 - 1.808


In [16]:
# Education over time
if 'educ' in available_vars and 'year' in gss.columns:
    educ_by_year = gss.groupby('year')['educ'].agg(['mean', 'std', 'count']).reset_index()
    
    fig = px.line(educ_by_year, x='year', y='mean',
                  title='Average Years of Education Over Time',
                  labels={'year': 'Year', 'mean': 'Average Years of Education'},
                  markers=True)
    fig.show()

In [17]:
# Relative income perception over time
if 'finrela' in available_vars and 'year' in gss.columns:
    finrela_by_year = gss.groupby('year')['finrela'].agg(['mean', 'count']).reset_index()
    finrela_by_year = finrela_by_year[finrela_by_year['count'] >= 50]
    
    if len(finrela_by_year) > 0:
        fig = px.line(finrela_by_year, x='year', y='mean',
                      title='Average Relative Income Perception Over Time (3=Average)',
                      labels={'year': 'Year', 'mean': 'Average Relative Income'},
                      markers=True)
        fig.add_hline(y=3, line_dash='dash', line_color='gray')
        fig.show()

## 6. Variable Relationships

In [18]:
# Correlation matrix for available target variables
corr_vars = [v for v in analysis_vars if v in gss.columns and gss[v].notna().sum() > 1000]

if len(corr_vars) > 2:
    # Compute pairwise correlations
    corr_data = gss[corr_vars].copy()
    corr_matrix = corr_data.corr()
    
    fig = px.imshow(corr_matrix,
                    title='Correlation Matrix (Pairwise Complete)',
                    color_continuous_scale='RdBu',
                    zmin=-1, zmax=1,
                    aspect='auto',
                    text_auto='.2f')
    fig.update_layout(height=600, width=700)
    fig.show()
    
    print(f"Variables in correlation matrix: {corr_vars}")

Variables in correlation matrix: ['satjob', 'hlthdep', 'feeldown', 'nointerest', 'feelnerv', 'worry', 'wrkmeangfl', 'stress', 'discaffwnv', 'wkageism', 'wkracism', 'wksexism', 'finrela', 'educ', 'age', 'sex']


In [19]:
# Correlations with job satisfaction
if 'satjob' in available_vars:
    print("Correlations with Job Satisfaction (SATJOB):")
    print("="*60)
    print("(Positive = higher values associated with more dissatisfaction)")
    print()
    
    correlations = []
    for var in analysis_vars:
        if var != 'satjob' and var in gss.columns:
            pair = gss[['satjob', var]].dropna()
            if len(pair) >= 50:
                r = pair['satjob'].corr(pair[var])
                correlations.append({
                    'Variable': var,
                    'Description': available_vars.get(var, var),
                    'Correlation': r,
                    'N': len(pair)
                })
    
    if correlations:
        corr_df = pd.DataFrame(correlations).sort_values('Correlation', key=abs, ascending=False)
        
        for _, row in corr_df.iterrows():
            print(f"{row['Variable']:<15} r = {row['Correlation']:>6.3f}  (n = {row['N']:,})")
        
        # Plot
        fig = px.bar(corr_df, x='Variable', y='Correlation',
                     title='Correlations with Job Satisfaction',
                     color='Correlation',
                     color_continuous_scale='RdBu',
                     color_continuous_midpoint=0)
        fig.update_layout(xaxis_tickangle=-45)
        fig.show()

Correlations with Job Satisfaction (SATJOB):
(Positive = higher values associated with more dissatisfaction)

wrkmeangfl      r =  0.506  (n = 1,950)
hlthdep         r =  0.341  (n = 751)
nointerest      r =  0.234  (n = 1,937)
feeldown        r =  0.230  (n = 1,936)
feelnerv        r =  0.228  (n = 1,937)
worry           r =  0.187  (n = 1,934)
stress          r = -0.174  (n = 13,289)
finrela         r = -0.159  (n = 53,338)
age             r = -0.149  (n = 53,568)
wkracism        r = -0.102  (n = 9,510)
wksexism        r = -0.099  (n = 8,363)
wkageism        r = -0.097  (n = 9,520)
discaffwnv      r = -0.059  (n = 1,793)
educ            r = -0.055  (n = 53,955)
sex             r = -0.003  (n = 54,035)


In [20]:
# Job satisfaction by key groups
if 'satjob' in available_vars:
    fig = make_subplots(rows=1, cols=2, subplot_titles=['By Sex', 'By Education Level'])
    
    # By sex
    if 'sex' in available_vars:
        satjob_sex = gss.groupby('sex')['satjob'].mean()
        fig.add_trace(
            go.Bar(x=['Male', 'Female'], y=satjob_sex.values, showlegend=False),
            row=1, col=1
        )
    
    # By education (binned)
    if 'educ' in available_vars:
        gss['educ_bin'] = pd.cut(gss['educ'], bins=[0, 11, 12, 15, 16, 20], 
                                 labels=['<HS', 'HS', 'Some College', 'Bachelor', 'Graduate'])
        satjob_educ = gss.groupby('educ_bin', observed=True)['satjob'].mean()
        fig.add_trace(
            go.Bar(x=satjob_educ.index.astype(str), y=satjob_educ.values, showlegend=False),
            row=1, col=2
        )
    
    fig.update_layout(height=400, title='Average Job Satisfaction by Group (lower = more satisfied)')
    fig.update_yaxes(title_text='Avg Job Satisfaction', row=1, col=1)
    fig.show()

## 7. Mental Health Variables Analysis

These variables are from more recent survey years and measure PHQ-4 depression/anxiety symptoms.

In [26]:
# Correlation among mental health variables
mh_vars = ['hlthdep', 'feeldown', 'nointerest', 'feelnerv', 'worry']
mh_available = [v for v in mh_vars if v in gss.columns and gss[v].notna().sum() > 100]

if len(mh_available) > 2:
    mh_corr = gss[mh_available].corr()
    
    fig = px.imshow(mh_corr,
                    title='Correlations Among Mental Health Variables',
                    color_continuous_scale='Blues',
                    zmin=0, zmax=1,
                    text_auto='.2f')
    fig.show()
    
    print("\nThese variables are highly correlated, suggesting they measure related constructs.")
    print("This motivates using regularization (horseshoe priors) in the regression model.")


These variables are highly correlated, suggesting they measure related constructs.
This motivates using regularization (horseshoe priors) in the regression model.


## 8. Complete Cases Analysis

For multivariate analysis, we need observations with all variables present.

In [23]:
# Check complete cases for different variable sets
print("Complete Cases Analysis:")
print("="*60)

# Core variables (from horseshoe model)
core_vars = ['satjob', 'hlthdep', 'feeldown', 'nointerest', 'feelnerv', 'worry', 'wrkmeangfl', 'age', 'sex']
core_available = [v for v in core_vars if v in gss.columns]
core_complete = gss[core_available].dropna().shape[0]
print(f"Core model variables ({len(core_available)} vars): {core_complete:,} complete cases")

# Extended with discrimination
discrim_vars = ['discaffwnv', 'discaffmv', 'wkageism', 'wkracism', 'wksexism']
discrim_available = [v for v in discrim_vars if v in gss.columns]
if discrim_available:
    ext_vars = core_available + discrim_available
    ext_complete = gss[ext_vars].dropna().shape[0]
    print(f"+ Discrimination vars ({len(ext_vars)} vars): {ext_complete:,} complete cases")

# With socioeconomic
se_vars = ['finrela', 'educ']
se_available = [v for v in se_vars if v in gss.columns]
if se_available:
    se_ext = core_available + se_available
    se_complete = gss[se_ext].dropna().shape[0]
    print(f"+ Socioeconomic vars ({len(se_ext)} vars): {se_complete:,} complete cases")

Complete Cases Analysis:
Core model variables (9 vars): 554 complete cases
+ Discrimination vars (13 vars): 0 complete cases
+ Socioeconomic vars (11 vars): 549 complete cases


In [24]:
# Focus on recent years where mental health variables were asked
if 'year' in gss.columns and 'hlthdep' in gss.columns:
    recent_years = gss.loc[gss['hlthdep'].notna(), 'year'].unique()
    print(f"\nMental health variables available in: {sorted(recent_years)}")
    
    for year in sorted(recent_years):
        year_data = gss[gss['year'] == year]
        n_total = len(year_data)
        n_complete = year_data[core_available].dropna().shape[0]
        print(f"  {int(year)}: {n_complete:,} complete cases out of {n_total:,} respondents")


Mental health variables available in: [np.int16(2022)]
  2022: 554 complete cases out of 3,544 respondents
